# Cross-Correlogram Analysis — M25 D25

Implements the rigorous monosynaptic detection method from English et al. / Stark & Abeles.

**Method:**
- CCG: 1 ms bins, ±50 ms window
- Baseline: hollow Gaussian convolution (σ=5 ms, hollow fraction=60%)
- Minimum criteria: ≥2000 spikes per cell, ≥1000 raw CCG counts
- **Five conditions** for a putative excitatory monosynaptic connection:
  1. Peak within 0.7–4.7 ms
  2. Peak height > 5 SD of baseline-corrected CCG
  3. Poisson p-value < 0.001 (with continuity correction)
  4. Peak width < 3 ms (contiguous bins > half-peak-height or 2 SD, P<0.01)
  5. Peak width does not overlap the zero-lag bin
- **Rejection**: any non-peak bin > 2.5 SD, or anticausal bins (negative lag) with P<0.01

In [ ]:
import numpy as np
import pandas as pd
import pynapple as nap
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from scipy.signal import fftconvolve
from scipy.stats import norm as sp_norm, poisson as sp_poisson
from spatial_manifolds.detect_grids import *

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
%matplotlib inline

plt.rcParams['font.family'] = 'Arial'

mouse       = 25
day         = 25
source_path = '/Users/harryclark/Downloads/COHORT12/'
fig_path    = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_playground/'

# ── CCG parameters (English / Stark & Abeles method) ─────────────────────────
BIN_MS          = 1        # 1 ms bins
MAX_LAG_MS      = 50       # ±50 ms window
SIGMA_MS        = 5        # hollow Gaussian σ
HOLLOW_FRAC     = 0.60     # fraction of Gaussian mass set to zero
MIN_SPIKES      = 2000     # minimum spikes per cell
MIN_CCG_COUNTS  = 1000     # minimum total counts in raw CCG
MONO_LO_MS      = 0.7      # monosynaptic window lower bound
MONO_HI_MS      = 4.7      # monosynaptic window upper bound
PEAK_Z_THRESH   = 5.0      # condition 2: peak > N SD
PEAK_P_THRESH   = 0.001    # condition 3: Poisson p-value
PEAK_W_THRESH   = 3.0      # condition 4: max peak width (ms)
WIDTH_P_THRESH  = 0.01     # condition 4: p-value for width bins
NONPEAK_Z       = 2.5      # rejection: non-peak bins > this SD
ANTICAUSAL_P    = 0.01     # rejection: anticausal bins p-value

COL_GC  = '#c04744'
COL_NGS = '#3171ae'
COL_NS  = '#aaaaaa'

cell_class = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/cell_classifications.csv')
sess_cells = cell_class[
    (cell_class['mouse'] == mouse) & (cell_class['day'] == day)
].copy()
sess_cells['cluster_id'] = sess_cells['cluster_id'].astype(int)
print(f'M{mouse} D{day}: {len(sess_cells)} classified cells')

## 1. Load spike times at 1 ms precision

In [ ]:
# Load spike clusters from the VR session (spike times in seconds)
vr_folder   = f'{source_path}M{mouse}/D{day:02}/VR/'
spikes_path = vr_folder + f'sub-{mouse}_day-{day:02}_ses-VR_srt-kilosort4_clusters.npz'

clusters_raw = nap.load_file(spikes_path)
clusters     = curate_clusters(clusters_raw)
cluster_ids  = list(clusters.index)
N            = len(cluster_ids)

# Duration from last spike
all_spk = np.concatenate([np.array(clusters[cid].t) for cid in cluster_ids])
T_s     = float(all_spk.max()) + 0.001
T_ms    = int(T_s * 1000)

print(f'Cells after curation: {N}')
print(f'Session length: {T_s:.1f} s  ({T_ms:,} ms bins at {BIN_MS} ms resolution)')

# Bin spike trains at BIN_MS resolution
print('Binning spike trains...')
spike_trains = np.zeros((N, T_ms), dtype=np.int8)
for i, cid in enumerate(cluster_ids):
    t_ms = (np.array(clusters[cid].t) * 1000 / BIN_MS).astype(int)
    t_ms = t_ms[(t_ms >= 0) & (t_ms < T_ms)]
    np.add.at(spike_trains[i], t_ms, 1)

n_spikes = spike_trains.sum(axis=1)
frates   = n_spikes / T_s
print(f'Mean firing rate: {frates.mean():.2f} Hz  (range {frates.min():.2f}–{frates.max():.2f} Hz)')

# Cell type labels
id_to_type = sess_cells.set_index('cluster_id')['cell_type'].to_dict()
cell_types = [id_to_type.get(cid, 'Other') for cid in cluster_ids]

## 2. Compute pairwise CCGs (FFT convolution)

In [ ]:
lags_ms = np.arange(-MAX_LAG_MS, MAX_LAG_MS + BIN_MS, BIN_MS, dtype=float)
N_lags  = len(lags_ms)
center  = MAX_LAG_MS   # index of lag=0

# ── Hollow Gaussian kernel ────────────────────────────────────────────────────
# σ=5ms, hollow fraction=60%: zero central bins carrying 60% of Gaussian mass
hollow_half = int(round(sp_norm.ppf((1 + HOLLOW_FRAC) / 2) * SIGMA_MS))
g = np.exp(-0.5 * (lags_ms / SIGMA_MS)**2)
g[center - hollow_half : center + hollow_half + 1] = 0.0
g /= g.sum()
print(f'Hollow Gaussian: σ={SIGMA_MS}ms, hollow half-width={hollow_half}ms  '
      f'(zeros lags {-hollow_half}..{hollow_half}ms)')

# ── Compute raw CCGs in spike COUNTS (needed for Poisson test) ────────────────
print(f'Computing {N*(N-1)//2} CCG pairs (±{MAX_LAG_MS}ms, {BIN_MS}ms bins)...')
# Store as float32 to manage memory; values are counts
ccg_raw  = np.zeros((N, N, N_lags), dtype=np.float32)  # raw counts
ccg_base = np.zeros((N, N, N_lags), dtype=np.float32)  # hollow-Gaussian baseline
ccg_corr = np.zeros((N, N, N_lags), dtype=np.float32)  # baseline-corrected

for i in range(N):
    for j in range(i + 1, N):
        # Full cross-correlation in counts
        full = fftconvolve(
            spike_trains[i].astype(np.float32),
            spike_trains[j][::-1].astype(np.float32),
            mode='full'
        )
        raw_ij = full[T_ms - 1 - MAX_LAG_MS : T_ms - 1 + MAX_LAG_MS + 1]

        # Baseline via hollow Gaussian convolution
        base_ij = np.convolve(raw_ij, g, mode='same')

        # Store both directions
        ccg_raw[i, j]  = raw_ij;          ccg_raw[j, i]  = raw_ij[::-1]
        ccg_base[i, j] = base_ij;         ccg_base[j, i] = base_ij[::-1]
        ccg_corr[i, j] = raw_ij - base_ij; ccg_corr[j, i] = (raw_ij - base_ij)[::-1]

    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{N} done', end='\r')

print(f'\nCCG matrix computed.')

## 3. Detect significant peaks — putative monosynaptic connections

In [ ]:
# ── Boolean masks ────────────────────────────────────────────────────────────
mono_mask     = (lags_ms >= MONO_LO_MS) & (lags_ms <= MONO_HI_MS)
anticausal_mask = lags_ms < 0
# Exclude monosynaptic window from SD estimate
sd_ref_mask   = ~mono_mask  # use all other bins for SD

# ── Poisson p-value helper (with continuity correction) ───────────────────────
def poisson_pval(count, expected):
    """P(X >= count - 0.5) under Poisson(expected), continuity corrected."""
    if expected <= 0:
        return 1.0
    return 1.0 - sp_poisson.cdf(count - 1.5, mu=expected)

connections = []
pairs_tested = 0
peak_z_matrix = np.zeros((N, N))

for i in range(N):
    for j in range(N):
        if i == j:
            continue
        if n_spikes[i] < MIN_SPIKES or n_spikes[j] < MIN_SPIKES:
            continue
        raw   = ccg_raw[i, j]
        base  = ccg_base[i, j]
        corr  = ccg_corr[i, j]
        if raw.sum() < MIN_CCG_COUNTS:
            continue
        pairs_tested += 1

        # SD from baseline-corrected CCG (exclude monosynaptic window)
        sigma = corr[sd_ref_mask].std()
        if sigma < 1e-10:
            continue

        # ── Find peak in monosynaptic window ──────────────────────────────────
        mono_corr = corr[mono_mask]
        if len(mono_corr) == 0:
            continue
        peak_idx_local = np.argmax(mono_corr)
        peak_idx_global = np.where(mono_mask)[0][peak_idx_local]
        peak_lag  = lags_ms[peak_idx_global]
        peak_corr = corr[peak_idx_global]
        peak_raw  = raw[peak_idx_global]
        peak_base = base[peak_idx_global]
        z_peak    = peak_corr / sigma
        peak_z_matrix[i, j] = z_peak

        # ── Condition 1: peak in 0.7–4.7 ms ──────────────────────────────────
        if not (MONO_LO_MS <= peak_lag <= MONO_HI_MS):
            continue

        # ── Condition 2: peak > PEAK_Z_THRESH SD ──────────────────────────────
        if z_peak < PEAK_Z_THRESH:
            continue

        # ── Condition 3: Poisson p < PEAK_P_THRESH ────────────────────────────
        p_peak = poisson_pval(peak_raw, peak_base)
        if p_peak >= PEAK_P_THRESH:
            continue

        # ── Condition 4: width < PEAK_W_THRESH ms ────────────────────────────
        # Width = contiguous bins touching the peak with:
        #   value > max(half_peak_height, 2*sigma) AND Poisson p < WIDTH_P_THRESH
        half_height = peak_corr / 2
        width_thresh = max(half_height, 2 * sigma)
        # Grow region from peak outward
        width_bins = set([peak_idx_global])
        for direction in [-1, 1]:
            idx = peak_idx_global + direction
            while 0 <= idx < N_lags:
                p_bin = poisson_pval(raw[idx], base[idx])
                if corr[idx] > width_thresh and p_bin < WIDTH_P_THRESH:
                    width_bins.add(idx)
                    idx += direction
                else:
                    break
        peak_width_ms = len(width_bins) * BIN_MS
        if peak_width_ms >= PEAK_W_THRESH:
            continue

        # ── Condition 5: width does not overlap zero-lag bin ──────────────────
        zero_idx = center   # index of lag=0
        if zero_idx in width_bins:
            continue

        # ── Rejection 1: any non-peak bin > NONPEAK_Z SD ─────────────────────
        non_peak_mask = np.ones(N_lags, dtype=bool)
        for idx in width_bins:
            non_peak_mask[idx] = False
        if np.any(corr[non_peak_mask] > NONPEAK_Z * sigma):
            continue

        # ── Rejection 2: anticausal bins with p < ANTICAUSAL_P ────────────────
        anticausal_rejected = False
        for idx in np.where(anticausal_mask)[0]:
            if poisson_pval(raw[idx], base[idx]) < ANTICAUSAL_P:
                anticausal_rejected = True
                break
        if anticausal_rejected:
            continue

        # ── All conditions passed: record connection ───────────────────────────
        connections.append(dict(
            pre=i, post=j,
            pre_id=cluster_ids[i], post_id=cluster_ids[j],
            peak_lag_ms=float(peak_lag),
            peak_z=float(z_peak),
            peak_p=float(p_peak),
            peak_width_ms=float(peak_width_ms),
        ))

n_total_pairs = N * (N - 1)
p_conn = len(connections) / pairs_tested if pairs_tested > 0 else 0
print(f'Pairs tested (met spike count criteria): {pairs_tested} / {n_total_pairs}')
print(f'Putative monosynaptic connections: {len(connections)}')
print(f'Overall connection probability: {p_conn:.4f}  ({100*p_conn:.2f}%)')
print()
for c in sorted(connections, key=lambda x: -x['peak_z']):
    pre_t  = cell_types[c['pre']]
    post_t = cell_types[c['post']]
    print(f'  {c["pre_id"]:4d}({pre_t}) → {c["post_id"]:4d}({post_t})  '
          f'lag={c["peak_lag_ms"]:+.1f}ms  '
          f'z={c["peak_z"]:.1f}  '
          f'p={c["peak_p"]:.1e}  '
          f'width={c["peak_width_ms"]:.0f}ms')

In [ ]:
# ── Top contenders by peak z-score (regardless of whether they pass all criteria) ─
N_TOP = 20   # number of pairs to inspect

# Collect all directed pairs with their z-scores and which conditions they fail
candidates = []
for i in range(N):
    for j in range(N):
        if i == j: continue
        if n_spikes[i] < MIN_SPIKES or n_spikes[j] < MIN_SPIKES: continue
        if ccg_raw[i, j].sum() < MIN_CCG_COUNTS: continue
        z = peak_z_matrix[i, j]
        if z <= 0: continue
        candidates.append((z, i, j))

candidates.sort(reverse=True)
top = candidates[:N_TOP]

print(f'Top {N_TOP} pairs by monosynaptic peak z-score (among tested pairs):')
print(f'{"Rank":>4}  {"Pre":>6}  {"Post":>6}  {"z":>6}  {"lag(ms)":>8}  {"Passes?"}')
print('-' * 60)
for rank, (z, i, j) in enumerate(top, 1):
    # Check which conditions fail
    raw   = ccg_raw[i, j]
    base  = ccg_base[i, j]
    corr  = ccg_corr[i, j]
    sigma = corr[sd_ref_mask].std()
    mono_corr = corr[mono_mask]
    peak_idx_local = np.argmax(mono_corr)
    peak_idx_global = np.where(mono_mask)[0][peak_idx_local]
    peak_lag = lags_ms[peak_idx_global]
    peak_raw = raw[peak_idx_global]
    peak_base = base[peak_idx_global]
    p_peak   = poisson_pval(peak_raw, peak_base)

    fails = []
    if not (MONO_LO_MS <= peak_lag <= MONO_HI_MS): fails.append('C1-lag')
    if z < PEAK_Z_THRESH:                           fails.append(f'C2-z<{PEAK_Z_THRESH}')
    if p_peak >= PEAK_P_THRESH:                     fails.append('C3-p')
    # Check anticausal
    if any(poisson_pval(raw[k], base[k]) < ANTICAUSAL_P
           for k in np.where(anticausal_mask)[0]):
        fails.append('R2-anticausal')
    # Check non-peak
    non_peak_vals = corr[~mono_mask]
    if np.any(non_peak_vals > NONPEAK_Z * sigma):
        fails.append(f'R1-nonpeak>{NONPEAK_Z}SD')

    status = 'PASS ✓' if not fails else 'FAIL: ' + ', '.join(fails)
    pre_t  = cell_types[i]; post_t = cell_types[j]
    print(f'{rank:>4}  {cluster_ids[i]:>4}({pre_t[0]})  '
          f'{cluster_ids[j]:>4}({post_t[0]})  '
          f'{z:>6.2f}  {peak_lag:>+8.1f}ms  {status}')

# ── Plot CCG traces for top contenders ────────────────────────────────────────
ncols = min(5, len(top))
nrows = int(np.ceil(len(top) / ncols))
fig, axes = plt.subplots(nrows, ncols,
                          figsize=(ncols * 3.0, nrows * 2.5),
                          gridspec_kw={'hspace': 0.6, 'wspace': 0.35})
axes_flat = np.array(axes).flatten()

for ai, (z, i, j) in enumerate(top):
    ax   = axes_flat[ai]
    raw  = ccg_raw[i, j]
    base = ccg_base[i, j]
    corr = ccg_corr[i, j]
    sigma = corr[sd_ref_mask].std()

    ax.bar(lags_ms, raw, width=BIN_MS * 0.85, color='#bbbbbb', alpha=0.8)
    ax.plot(lags_ms, base, color='#e67e22', lw=1.2)
    ax.axhline(base.mean() + PEAK_Z_THRESH * sigma,
               color='#e74c3c', lw=0.8, ls='--', alpha=0.7)
    ax.axhline(base.mean() + 3 * sigma,
               color='#3498db', lw=0.8, ls=':', alpha=0.7)
    ax.axvspan(MONO_LO_MS, MONO_HI_MS, color='#e74c3c', alpha=0.10)

    # Mark the monosynaptic peak
    peak_idx_global = np.where(mono_mask)[0][np.argmax(corr[mono_mask])]
    ax.bar(lags_ms[peak_idx_global], raw[peak_idx_global],
           width=BIN_MS * 0.85, color='#e74c3c', alpha=0.9)

    ax.axvline(0, color='k', lw=0.5, alpha=0.4)
    pre_t  = cell_types[i]; post_t = cell_types[j]
    ax.set_title(
        f'#{ai+1}  {cluster_ids[i]}({pre_t[0]})→{cluster_ids[j]}({post_t[0]})\n'
        f'z={z:.1f}  lag={lags_ms[peak_idx_global]:+.1f}ms',
        fontsize=7
    )
    ax.set_xlim(-MAX_LAG_MS - 1, MAX_LAG_MS + 1)
    ax.set_xlabel('Lag (ms)', fontsize=6)
    ax.set_ylabel('Counts', fontsize=6)
    ax.tick_params(labelsize=5.5)
    ax.spines[['top', 'right']].set_visible(False)

for ai in range(len(top), len(axes_flat)):
    axes_flat[ai].axis('off')

fig.suptitle(
    f'M{mouse} D{day} — Top {N_TOP} pairs by peak z-score in 0.7–4.7 ms window\n'
    f'Red bars = peak bin  |  Orange = hollow-Gaussian baseline  |  '
    f'Dashed red = {PEAK_Z_THRESH} SD  |  Dotted blue = 3 SD',
    fontsize=8, fontweight='bold'
)
plt.savefig(fig_path + f'ccg_top_contenders_M{mouse}D{day}.pdf',
            bbox_inches='tight', dpi=200)
plt.show()
print(f'\nTo relax criteria, reduce PEAK_Z_THRESH (currently {PEAK_Z_THRESH}) '
      f'or PEAK_P_THRESH (currently {PEAK_P_THRESH})')
print(f'Or widen the monosynaptic window ({MONO_LO_MS}–{MONO_HI_MS} ms)')


In [ ]:
# ── Detailed diagnostic: exactly what each top pair fails and by how much ─────
N_DIAG = min(10, len(top))
print(f'Detailed criteria breakdown for top {N_DIAG} pairs\n')
print(f'Thresholds:  C1 lag=[{MONO_LO_MS},{MONO_HI_MS}]ms  '
      f'C2 z>{PEAK_Z_THRESH}  C3 p<{PEAK_P_THRESH}  '
      f'C4 width<{PEAK_W_THRESH}ms  R1 nonpeak<{NONPEAK_Z}SD  '
      f'R2 anticausal p>{ANTICAUSAL_P}')
print('─'*80)

for rank, (z, i, j) in enumerate(top[:N_DIAG], 1):
    raw   = ccg_raw[i, j]
    base  = ccg_base[i, j]
    corr  = ccg_corr[i, j]
    sigma = corr[sd_ref_mask].std()

    peak_idx_global = np.where(mono_mask)[0][np.argmax(corr[mono_mask])]
    peak_lag   = lags_ms[peak_idx_global]
    peak_raw   = float(raw[peak_idx_global])
    peak_base  = float(base[peak_idx_global])
    p_peak     = poisson_pval(peak_raw, peak_base)

    # C4: peak width
    half_height  = corr[peak_idx_global] / 2
    width_thresh = max(half_height, 2 * sigma)
    width_bins   = {peak_idx_global}
    for direction in [-1, 1]:
        idx = peak_idx_global + direction
        while 0 <= idx < N_lags:
            p_bin = poisson_pval(float(raw[idx]), float(base[idx]))
            if corr[idx] > width_thresh and p_bin < WIDTH_P_THRESH:
                width_bins.add(idx); idx += direction
            else:
                break
    peak_width_ms = len(width_bins) * BIN_MS
    overlaps_zero = (np.where(lags_ms == 0)[0][0]) in width_bins

    # R1: worst non-peak bin
    non_peak_corr = corr.copy(); non_peak_corr[list(width_bins)] = 0
    worst_nonpeak_z = float(non_peak_corr.max() / sigma)
    worst_nonpeak_lag = float(lags_ms[np.argmax(non_peak_corr)])

    # R2: worst anticausal p-value
    ac_pvals = [poisson_pval(float(raw[k]), float(base[k]))
                for k in np.where(anticausal_mask)[0]]
    worst_ac_p = float(min(ac_pvals)) if ac_pvals else 1.0
    worst_ac_lag = float(lags_ms[anticausal_mask][np.argmin(ac_pvals)]) if ac_pvals else 0

    pre_t  = cell_types[i]; post_t = cell_types[j]
    print(f'\n#{rank}  {cluster_ids[i]}({pre_t}) → {cluster_ids[j]}({post_t})')
    print(f'   C1  peak lag       = {peak_lag:+.1f} ms   '
          f'[need {MONO_LO_MS}–{MONO_HI_MS}ms]  '
          f'{"PASS" if MONO_LO_MS<=peak_lag<=MONO_HI_MS else "FAIL"}')
    print(f'   C2  peak z-score   = {z:.2f}            '
          f'[need >{PEAK_Z_THRESH}]           '
          f'{"PASS" if z>=PEAK_Z_THRESH else "FAIL"}')
    print(f'   C3  Poisson p      = {p_peak:.2e}       '
          f'[need <{PEAK_P_THRESH}]      '
          f'{"PASS" if p_peak<PEAK_P_THRESH else "FAIL"}')
    print(f'   C4  peak width     = {peak_width_ms:.0f} ms             '
          f'[need <{PEAK_W_THRESH}ms]          '
          f'{"PASS" if peak_width_ms<PEAK_W_THRESH else "FAIL"}')
    print(f'   C5  overlaps zero  = {str(overlaps_zero):5}            '
          f'[need False]         '
          f'{"PASS" if not overlaps_zero else "FAIL"}')
    print(f'   R1  worst non-peak = {worst_nonpeak_z:.2f} SD  '
          f'at lag {worst_nonpeak_lag:+.0f}ms  '
          f'[need <{NONPEAK_Z}SD]  '
          f'{"PASS" if worst_nonpeak_z<NONPEAK_Z else "FAIL"}')
    print(f'   R2  worst anticaus = p={worst_ac_p:.2e}  '
          f'at lag {worst_ac_lag:+.0f}ms  '
          f'[need >{ANTICAUSAL_P}]    '
          f'{"PASS" if worst_ac_p>=ANTICAUSAL_P else "FAIL"}')


## 4. N×N CCG summary heatmap

In [ ]:
fig_hm, ax = plt.subplots(figsize=(8, 7))

z_display = np.clip(peak_z_matrix, 0, PEAK_Z_THRESH * 2)
np.fill_diagonal(z_display, 0)

im = ax.imshow(z_display, cmap='hot', vmin=0, vmax=PEAK_Z_THRESH * 2,
               aspect='auto', interpolation='nearest', origin='upper')
cb = plt.colorbar(im, ax=ax, fraction=0.04)
cb.set_label(f'Peak z-score (baseline-corrected, 0.7–4.7 ms lag)', fontsize=8)
cb.ax.axhline(PEAK_Z_THRESH, color='cyan', lw=1.5)

# Mark significant connections
for c in connections:
    ax.plot(c['post'], c['pre'], 'c.', ms=7, zorder=5)

type_color = {'GC': COL_GC, 'NG': COL_NGS, 'Other': COL_NS}
ax.set_xticks(range(N)); ax.set_yticks(range(N))
ax.set_xticklabels([str(cid) for cid in cluster_ids], rotation=90, fontsize=4)
ax.set_yticklabels([str(cid) for cid in cluster_ids], fontsize=4)
for tick, ct in zip(ax.get_xticklabels(), cell_types):
    tick.set_color(type_color.get(ct, COL_NS))
for tick, ct in zip(ax.get_yticklabels(), cell_types):
    tick.set_color(type_color.get(ct, COL_NS))

ax.set_xlabel('Post-synaptic cell ID', fontsize=9)
ax.set_ylabel('Pre-synaptic cell ID', fontsize=9)
ax.set_title(
    f'M{mouse} D{day} — CCG peak z-score matrix (hollow-Gaussian baseline)\n'
    f'Cyan = z > {PEAK_Z_THRESH} at 0.7–4.7 ms  |  '
    f'tick colour: red=GC  blue=NGS  grey=Other',
    fontsize=9)

plt.tight_layout()
plt.savefig(fig_path + f'ccg_matrix_M{mouse}D{day}.pdf', bbox_inches='tight', dpi=300)
plt.show()

## 5. Individual CCG traces for connected pairs

In [ ]:
if not connections:
    print('No significant connections found.')
else:
    n_conn = len(connections)
    ncols  = min(4, n_conn)
    nrows  = int(np.ceil(n_conn / ncols))

    fig_ccg, axes = plt.subplots(
        nrows, ncols, figsize=(ncols * 3.0, nrows * 2.5),
        gridspec_kw={'hspace': 0.55, 'wspace': 0.35}
    )
    if n_conn == 1: axes = np.array([[axes]])
    axes_flat = np.array(axes).flatten()

    for ai, c in enumerate(sorted(connections, key=lambda x: -x['peak_z'])):
        ax   = axes_flat[ai]
        i, j = c['pre'], c['post']
        raw  = ccg_raw[i, j]
        base = ccg_base[i, j]
        corr = ccg_corr[i, j]
        sigma = corr[sd_ref_mask].std()

        # Raw CCG (grey bars)
        ax.bar(lags_ms, raw, width=BIN_MS * 0.85,
               color='#bbbbbb', alpha=0.8, label='Raw CCG')
        # Baseline (orange line)
        ax.plot(lags_ms, base, color='#e67e22', lw=1.2, label='Baseline')
        # Significance thresholds on corrected CCG (dashed)
        ax.axhline(base.mean() + PEAK_Z_THRESH * sigma,
                   color='#e74c3c', lw=0.8, ls='--',
                   label=f'{PEAK_Z_THRESH} SD')

        # Highlight monosynaptic window
        ax.axvspan(MONO_LO_MS, MONO_HI_MS, color='#e74c3c', alpha=0.10, zorder=0)

        # Mark peak bin
        peak_idx = np.where(mono_mask)[0][np.argmax(corr[mono_mask])]
        ax.bar(lags_ms[peak_idx], raw[peak_idx], width=BIN_MS * 0.85,
               color='#e74c3c', alpha=0.9, label='Peak')

        ax.axvline(0, color='k', lw=0.5, alpha=0.4)
        pre_t  = cell_types[i]
        post_t = cell_types[j]
        ax.set_title(
            f'{cluster_ids[i]}({pre_t})→{cluster_ids[j]}({post_t})\n'
            f'lag={c["peak_lag_ms"]:+.1f}ms  z={c["peak_z"]:.1f}  '
            f'p={c["peak_p"]:.1e}  w={c["peak_width_ms"]:.0f}ms',
            fontsize=7
        )
        ax.set_xlabel('Lag (ms)', fontsize=7)
        ax.set_ylabel('Spike counts', fontsize=7)
        ax.set_xlim(-MAX_LAG_MS - 1, MAX_LAG_MS + 1)
        ax.tick_params(labelsize=6)
        ax.spines[['top', 'right']].set_visible(False)
        if ai == 0:
            ax.legend(fontsize=5.5, frameon=False, loc='upper left')

    for ai in range(n_conn, len(axes_flat)):
        axes_flat[ai].axis('off')

    fig_ccg.suptitle(
        f'M{mouse} D{day} — Putative monosynaptic connections\n'
        f'(z>{PEAK_Z_THRESH}, lag 0.7–4.7 ms, p<{PEAK_P_THRESH}, '
        f'width<{PEAK_W_THRESH}ms, hollow-Gaussian baseline)',
        fontsize=9, fontweight='bold'
    )
    plt.savefig(fig_path + f'ccg_traces_M{mouse}D{day}.pdf',
                bbox_inches='tight', dpi=300)
    plt.show()

## 6. Probe map with monosynaptic edges

In [ ]:
# Get probe coordinates for all cells
id_to_probe = sess_cells.set_index('cluster_id')[['probe_x', 'probe_y']].to_dict('index')

fig_probe, ax = plt.subplots(figsize=(3.5, 9))

# ── Draw monosynaptic edges first (behind cell markers) ──────────────────────
for c in connections:
    pre_id  = cluster_ids[c['pre']]
    post_id = cluster_ids[c['post']]
    if pre_id not in id_to_probe or post_id not in id_to_probe:
        continue
    px0 = id_to_probe[pre_id]['probe_x']
    py0 = id_to_probe[pre_id]['probe_y']
    px1 = id_to_probe[post_id]['probe_x']
    py1 = id_to_probe[post_id]['probe_y']

    # Line width scaled by z-score, colour by lag
    lw    = np.clip((c['peak_z'] - SIG_THRESH) * 0.5 + 0.8, 0.5, 3.0)
    color = '#e74c3c'   # excitatory connections in red
    ax.annotate('',
        xy=(px1, py1), xytext=(px0, py0),
        arrowprops=dict(arrowstyle='->', color=color,
                        lw=lw, shrinkA=6, shrinkB=6))

# ── Plot cells ────────────────────────────────────────────────────────────────
type_color = {'GC': COL_GC, 'NG': COL_NGS, 'Other': COL_NS}
type_label = {'GC': 'Grid cell', 'NG': 'NGS cell', 'Other': 'Other'}

# Connected cell IDs
connected_ids = set()
for c in connections:
    connected_ids.add(cluster_ids[c['pre']])
    connected_ids.add(cluster_ids[c['post']])

for i, cid in enumerate(cluster_ids):
    if cid not in id_to_probe:
        continue
    px = id_to_probe[cid]['probe_x']
    py = id_to_probe[cid]['probe_y']
    ct = cell_types[i]
    col = type_color.get(ct, COL_NS)
    size = 60 if cid in connected_ids else 25
    edge = 'black' if cid in connected_ids else col
    ax.scatter(px, py, s=size, color=col, edgecolors=edge,
               linewidths=0.8 if cid in connected_ids else 0.3,
               zorder=3)

# ── Legend ────────────────────────────────────────────────────────────────────
legend_els = [
    plt.scatter([], [], s=25, color=COL_GC, label='Grid cell'),
    plt.scatter([], [], s=25, color=COL_NGS, label='NGS cell'),
    plt.scatter([], [], s=25, color=COL_NS,  label='Other'),
    plt.scatter([], [], s=60, color='white', edgecolors='black',
                linewidths=0.8, label='Connected cell'),
    Line2D([0],[0], color='#e74c3c', lw=1.5, marker='>',
           markersize=6, label=f'Excitatory (z>{SIG_THRESH}, 1–3 ms)'),
]
ax.legend(handles=legend_els, fontsize=7, frameon=False, loc='lower right')
ax.set_xlabel('Probe x (µm)', fontsize=9)
ax.set_ylabel('Probe y (µm)', fontsize=9)
ax.set_title(
    f'M{mouse} D{day}\nProbe map + monosynaptic connections\n'
    f'(CCG peak z > {SIG_THRESH} at 1–3 ms)',
    fontsize=9, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(labelsize=8)
plt.tight_layout()
plt.savefig(fig_path + f'ccg_probe_map_M{mouse}D{day}.pdf',
            bbox_inches='tight', dpi=300)
plt.show()
print(f'{len(connections)} edges drawn  |  {len(connected_ids)} cells involved')